# 01 — Exploratory Data Analysis: NASA C-MAPSS

Quick exploration of the FD001 subset of the C-MAPSS turbofan engine degradation dataset.

**Goal**: understand sensor dynamics, identify a 'healthy operation' regime to train the autoencoder on, and pick the features that actually carry information.

Prerequisites: run `make download-data` first.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make the src package importable when running the notebook from notebooks/.
sys.path.insert(0, str(Path.cwd().parent))

from src.data.loaders import load_cmapss

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 4)

In [ ]:
ds = load_cmapss(subset="FD001")
ds

## Basic stats

In [ ]:
print('Train:', ds.train.shape)
print('Test:', ds.test.shape)
print('Train RUL distribution:')
ds.train.groupby('unit')['cycle'].max().describe()

## Sensor trajectories for a few engines

Plotting the first six engines makes the degradation pattern visible: some sensors drift monotonically as the engine approaches failure, others remain noisy with no trend.

In [ ]:
units_to_plot = [1, 2, 3, 4, 5, 6]
sensors_to_plot = ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_11', 'sensor_15']

fig, axes = plt.subplots(len(sensors_to_plot), 1, figsize=(12, 2.4 * len(sensors_to_plot)), sharex=True)
for ax, sensor in zip(axes, sensors_to_plot):
    for u in units_to_plot:
        sub = ds.train[ds.train['unit'] == u]
        ax.plot(sub['cycle'], sub[sensor], alpha=0.6, label=f'unit {u}')
    ax.set_title(sensor)
    ax.set_ylabel(sensor)
axes[-1].set_xlabel('cycle')
axes[0].legend(ncol=6, fontsize=8)
plt.tight_layout()

## RUL distribution

In [ ]:
fig, ax = plt.subplots()
ds.train['RUL'].hist(bins=50, ax=ax)
ax.set_title('Training RUL distribution (FD001)')
ax.set_xlabel('Remaining Useful Life (cycles)')

## Next steps

1. Drop constant / low-information sensors (already handled by `select_feature_columns`).
2. Define the 'healthy' regime — first ~50 cycles of each engine.
3. Build sliding windows and train the LSTM autoencoder (see `scripts/train.py`).